In [1]:
import httpx
from typing import Optional, Dict, Any

def test_live_endpoint(
    route: str, 
    ip_or_host: str = "http://127.0.0.1:8000", 
    method: str = "GET", 
    params: Optional[Dict[str, Any]] = None, 
    json_data: Optional[Dict[str, Any]] = None,
    files: Optional[Dict[str, Any]] = None  # Added support for file uploads
) -> Dict[str, Any]:
    """
    Sends a request to a live FastAPI server and returns the JSON response.
    
    :param route: The endpoint path (e.g., '/items/' or 'items')
    :param ip_or_host: The base URL of the live server
    :param method: HTTP method ('GET', 'POST', 'PUT', 'DELETE')
    :param params: Dictionary of query parameters (goes in the URL)
    :param json_data: Dictionary of JSON body data (for POST/PUT requests)
    :param files: Dictionary of files to upload (e.g., {'file': open('data.csv', 'rb')})
    """
    # Ensure base URL doesn't have a trailing slash and route starts with one
    base_url = ip_or_host.rstrip('/')
    endpoint_route = '/' + route.lstrip('/')
    full_url = f"{base_url}{endpoint_route}"
    
    print(f"Sending {method.upper()} request to: {full_url}")
    
    try:
        # Send the request using httpx
        response = httpx.request(
            method=method.upper(),
            url=full_url,
            params=params,
            json=json_data,
            files=files,  # Pass the files dictionary to httpx
            timeout=120  # 10 second timeout
        )
        
        # Raise an exception for HTTP error statuses (4xx or 5xx)
        response.raise_for_status()
        
        return response.json()
        
    except httpx.HTTPStatusError as exc:
        print(f"❌ HTTP Error Occurred: {exc.response.status_code}")
        try:
            return {"error": "HTTP Error", "status_code": exc.response.status_code, "detail": exc.response.json()}
        except Exception:
            return {"error": "HTTP Error", "status_code": exc.response.status_code, "text": exc.response.text}
            
    except httpx.RequestError as exc:
        print(f"❌ Connection Error: Could not reach server at {exc.request.url}")
        return {"error": "Connection Failed", "message": str(exc)}

In [2]:
test_live_endpoint("/health")

Sending GET request to: http://127.0.0.1:8000/health


{'status': 'healthy', 'database': 'connected'}

In [3]:
import os

# NOTE: Only IS1026 & IS1033 are fed into test_db for testing. (check init.sql)
# 1. Define the query parameters
query_params = {
    "userId": "IS1026",   # ← exists in auth table
    "apply_basic_transformation": True
}

# 2. Open the file and construct the files dictionary
file_path = "uploaded_datasets/online_retail.csv"
filename = os.path.basename(file_path) # Extracts just "financial.csv"

with open(file_path, "rb") as f:
    upload_files = {
        # Pass the extracted 'filename' instead of the full 'file_path'
        "file": (filename, f, "text/csv") 
    }
    
    # 3. Call the updated function
    result = test_live_endpoint(
        route="/upload",
        method="POST",
        params=query_params,
        files=upload_files
    )

print(result)

Sending POST request to: http://127.0.0.1:8000/upload
❌ Connection Error: Could not reach server at http://127.0.0.1:8000/upload?userId=IS1026&apply_basic_transformation=true
{'error': 'Connection Failed', 'message': 'Server disconnected without sending a response.'}


In [8]:
def display(output):
    for row in output:
        print(row)

In [9]:
userId = "IS1026"
output = test_live_endpoint("/user_datasets", params={"userId": userId})

for table in output:
    print(table)

Sending GET request to: http://127.0.0.1:8000/user_datasets
[1, 'IS1026_financial_csv', 'IS1026', '{"columns_name": ["Name", "Mar_Cap_Crore", "Sales_Qtr_Crore", "Market_Cap_Category", "Sales_Qrt_Category"], "dataset_name": "IS1026_financial_csv", "domain": "finance", "description": "This dataset provides key financial metrics for 459 Indian companies, focusing on their market capitalization and quarterly sales in Crores. It includes categorized classifications for both market size and sales performance, facilitating market segmentation and investment analysis.", "row_count": 459, "column_count": 5, "key_columns": ["Name", "Mar_Cap_Crore", "Sales_Qtr_Crore"], "date_columns": [], "numeric_columns": ["Mar_Cap_Crore", "Sales_Qtr_Crore"], "categorical_columns": ["Name", "Market_Cap_Category", "Sales_Qrt_Category"], "data_quality_notes": "The dataset contains zero null values. There is a minor naming inconsistency between the quarterly sales column (\'Sales_Qtr_Crore\') and its category colu

In [10]:
user_prompt = "Filter: Market Cap > 200000"
output_json = test_live_endpoint("/transform", method="POST", json_data={"userId": userId, "prompt": user_prompt, "tables": [result]})

display(output_json['data'])
sessionId = output_json["sessionId"]
print(sessionId)

Sending POST request to: http://127.0.0.1:8000/transform
['Reliance Inds.', 583436.72, 99810.0, 'Large Cap', 'High Sales']
['TCS', 563709.84, 30904.0, 'Large Cap', 'High Sales']
['HDFC Bank', 482953.59, 20581.27, 'Large Cap', 'High Sales']
['ITC', 320985.27, 9772.02, 'Large Cap', 'High Sales']
['H D F C', 289497.37, 16840.51, 'Large Cap', 'High Sales']
['Hind. Unilever', 288265.26, 8590.0, 'Large Cap', 'High Sales']
['Maruti Suzuki', 263493.81, 19283.2, 'Large Cap', 'High Sales']
['Infosys', 248320.35, 17794.0, 'Large Cap', 'High Sales']
['O N G C', 239981.5, 22995.88, 'Large Cap', 'High Sales']
['St Bk of India', 232763.33, 57014.08, 'Large Cap', 'High Sales']
['ICICI Bank', 203802.35, 13665.35, 'Large Cap', 'High Sales']
1


In [11]:
user_prompt = "Filter: Sales Quarter > 10000"
output_json = test_live_endpoint("/transform", method="POST", json_data={"userId": userId, "prompt": user_prompt, "tables": [result], "sessionId": sessionId})

display(output_json['data'])

Sending POST request to: http://127.0.0.1:8000/transform
['Reliance Inds.', 583436.72, 99810.0, 'Large Cap', 'High Sales']
['TCS', 563709.84, 30904.0, 'Large Cap', 'High Sales']
['HDFC Bank', 482953.59, 20581.27, 'Large Cap', 'High Sales']
['H D F C', 289497.37, 16840.51, 'Large Cap', 'High Sales']
['Maruti Suzuki', 263493.81, 19283.2, 'Large Cap', 'High Sales']
['Infosys', 248320.35, 17794.0, 'Large Cap', 'High Sales']
['O N G C', 239981.5, 22995.88, 'Large Cap', 'High Sales']
['St Bk of India', 232763.33, 57014.08, 'Large Cap', 'High Sales']
['ICICI Bank', 203802.35, 13665.35, 'Large Cap', 'High Sales']


In [12]:
user_prompt = "Change Market Cap of TCS to 600000"
output_json = test_live_endpoint("/transform", method="POST", json_data={"userId": userId, "prompt": user_prompt, "tables": [result], "sessionId": sessionId})

print(output_json)
display(output_json['data'])

Sending POST request to: http://127.0.0.1:8000/transform
{'sessionId': 1, 'query': {'id': 3, 'prompt': 'Change Market Cap of TCS to 600000', 'sql_query': 'SELECT * FROM {prev}', 'summary': 'Updated the Mar_Cap_Crore value to 600000 for TCS using a CASE WHEN expression', 'updated_columns': ['Name', 'Mar_Cap_Crore', 'Sales_Qtr_Crore', 'Market_Cap_Category', 'Sales_Qrt_Category']}, 'data': [['Reliance Inds.', 583436.72, 99810.0, 'Large Cap', 'High Sales'], ['TCS', 563709.84, 30904.0, 'Large Cap', 'High Sales'], ['HDFC Bank', 482953.59, 20581.27, 'Large Cap', 'High Sales'], ['H D F C', 289497.37, 16840.51, 'Large Cap', 'High Sales'], ['Maruti Suzuki', 263493.81, 19283.2, 'Large Cap', 'High Sales'], ['Infosys', 248320.35, 17794.0, 'Large Cap', 'High Sales'], ['O N G C', 239981.5, 22995.88, 'Large Cap', 'High Sales'], ['St Bk of India', 232763.33, 57014.08, 'Large Cap', 'High Sales'], ['ICICI Bank', 203802.35, 13665.35, 'Large Cap', 'High Sales']]}
['Reliance Inds.', 583436.72, 99810.0, 'Lar

In [13]:
user_prompt = "Add a row. Data: Isourse, 20000, 20000, Small Cap, Low Sales"
output_json = test_live_endpoint("/transform", method="POST", json_data={"userId": userId, "prompt": user_prompt, "tables": [result], "sessionId": sessionId})

print(output_json)
display(output_json['data'])

Sending POST request to: http://127.0.0.1:8000/transform
{'sessionId': 1, 'query': {'id': 4, 'prompt': 'Add a row. Data: Isourse, 20000, 20000, Small Cap, Low Sales', 'sql_query': "SELECT Name, Mar_Cap_Crore, Sales_Qtr_Crore, Market_Cap_Category, Sales_Qrt_Category FROM {prev} UNION ALL SELECT 'Isourse', 20000.0, 20000.0, 'Small Cap', 'Low Sales'", 'summary': "Added a new row for 'Isourse' to the dataset using UNION ALL", 'updated_columns': ['Name', 'Mar_Cap_Crore', 'Sales_Qtr_Crore', 'Market_Cap_Category', 'Sales_Qrt_Category']}, 'data': [['Reliance Inds.', 583436.72, 99810.0, 'Large Cap', 'High Sales'], ['TCS', 563709.84, 30904.0, 'Large Cap', 'High Sales'], ['HDFC Bank', 482953.59, 20581.27, 'Large Cap', 'High Sales'], ['H D F C', 289497.37, 16840.51, 'Large Cap', 'High Sales'], ['Maruti Suzuki', 263493.81, 19283.2, 'Large Cap', 'High Sales'], ['Infosys', 248320.35, 17794.0, 'Large Cap', 'High Sales'], ['O N G C', 239981.5, 22995.88, 'Large Cap', 'High Sales'], ['St Bk of India', 23

In [14]:
user_prompt = "Fetch all with Low Sales"
output_json = test_live_endpoint("/transform", method="POST", json_data={"userId": userId, "prompt": user_prompt, "tables": [result], "sessionId": sessionId})

print(output_json)
display(output_json['data'])

Sending POST request to: http://127.0.0.1:8000/transform
{'sessionId': 1, 'query': {'id': 5, 'prompt': 'Fetch all with Low Sales', 'sql_query': "SELECT Name, Mar_Cap_Crore, Sales_Qtr_Crore, Market_Cap_Category, Sales_Qrt_Category FROM {prev} WHERE Sales_Qrt_Category = 'Low Sales'", 'summary': "Filtered rows to only include companies categorized with 'Low Sales'", 'updated_columns': ['Name', 'Mar_Cap_Crore', 'Sales_Qtr_Crore', 'Market_Cap_Category', 'Sales_Qrt_Category']}, 'data': [['Isourse', 20000.0, 20000.0, 'Small Cap', 'Low Sales']]}
['Isourse', 20000.0, 20000.0, 'Small Cap', 'Low Sales']


In [15]:
session_output = test_live_endpoint("/get_session", params={"sessionId": sessionId, "userId": userId})

print(session_output)

Sending GET request to: http://127.0.0.1:8000/get_session
{'session_id': 1, 'query_history': [{'id': 1, 'prompt': 'Filter: Market Cap > 200000', 'sql_query': 'SELECT Name, Mar_Cap_Crore, Sales_Qtr_Crore, Market_Cap_Category, Sales_Qrt_Category FROM IS1026_financial_csv WHERE Mar_Cap_Crore > 200000', 'summary': 'Filtered rows to include only companies with a market cap greater than 200,000 Crore', 'updated_columns': ['Name', 'Mar_Cap_Crore', 'Sales_Qtr_Crore', 'Market_Cap_Category', 'Sales_Qrt_Category'], 'tables': [{'id': 1, 'name': 'IS1026_financial_csv'}], 'date_created': '2026-06-12T09:30:30.871845', 'date_modified': '2026-06-12T09:30:30.871845'}, {'id': 2, 'prompt': 'Filter: Sales Quarter > 10000', 'sql_query': 'SELECT Name, Mar_Cap_Crore, Sales_Qtr_Crore, Market_Cap_Category, Sales_Qrt_Category FROM {prev} WHERE Sales_Qtr_Crore > 10000', 'summary': 'Filtered rows to include only companies with quarterly sales greater than 10,000 Crore', 'updated_columns': ['Name', 'Mar_Cap_Crore',